x# Compute soil CO2 from NPP and temperature

Reads a CSV of site coordinates (`lat`, `lon`), samples MODIS NPP, computes annual soil respiration with the RS92 empiraical relationship:

$$SR = 1.24 * NPP + 24.5$$

Where SR is total CO2 production in gC/m2/yr ([Raich and Schlesinger, 1992](https://doi.org/10.3402/tellusb.v44i2.15428)).

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import s3fs
import fsspec

from byte_util.util import all_sites

s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = "s3://carbonplan-carbon-removal/ew-workflows-data/min3p"

INPUT_CSV  = f"{s3_base_path}/sites/site-locations.csv"

redownload = False     # whether to redownload MODIS NPP or to use cached copy

### Read list of input sites

In [ ]:
df = pd.read_csv(INPUT_CSV)
assert {"lat", "lon"}.issubset(df.columns), "CSV must contain 'lat' and 'lon' columns"
print(f"{len(df)} sites loaded")

## Sample MODIS NPP at each site

In [ ]:
MODIS_YEARS = range(2000, 2016)

cached_file = Path('.cache/modis_NPP.parquet')

if redownload or not cached_file.exists():
    df = pd.read_csv(INPUT_CSV)

    xy = {"x": xr.DataArray(df["lon"].values, dims="points"),
          "y": xr.DataArray(df["lat"].values, dims="points")}

    npp_by_year = []

    for year in tqdm(MODIS_YEARS):
        url = ("http://files.ntsg.umt.edu/data/NTSG_Products/MOD17/GeoTIFF/"
               f"MOD17A3/GeoTIFF_30arcsec/MOD17A3_Science_NPP_{year}.tif")

        with xr.open_dataset(url, engine="rasterio", chunks="auto") as ds:
            npp = (ds["band_data"].isel(band=0).sel(**xy, method="nearest").compute()* 0.0001* 1e3)  # g C m-2 yr-1
        npp_by_year.append(npp.assign_coords(year=year))

    npp_by_year = xr.concat(npp_by_year, dim="year")
    npp_annual_df = pd.DataFrame(npp_by_year.values, columns=df['site_id'], index=npp_by_year["year"].values,)
    npp_annual_df.to_parquet(cached_file)
else:
    npp_annual_df = pd.read_parquet(cached_file)

# Add mean NPP to df
df["MODIS_NPP_gC_m2_yr"] = npp_annual_df.mean(axis=0).values

# Set site_id to be the index and sort on it
df.set_index('site_id', inplace=True)
df.sort_index(inplace=True)

print('MODIS annual NPP values:')
print(npp_annual_df.head())

## Convert NPP to soil CO2 using RS92 empirical relationship

In [ ]:
# Choose which soil respiration calculation to use: 'RS92' or 'GB94'
# and apply a scaling factor
soil_resp_calc = 'GB94'
for soil_resp_calc in ['GB94', 'RS92']:
    # Constants
    MOLAR_MASS_C = 12.011           # g C mol-1
    SECONDS_PER_YEAR = 365.25 * 24 * 60 * 60  # s yr-1

    # Raich and Schlesinger (1992)
    if soil_resp_calc == 'RS92':
        df["soil_respiration_gC_m2_yr"] = 1.24 * df["MODIS_NPP_gC_m2_yr"] + 24.5

    # Gwiazda and Broecker (1994)
    if soil_resp_calc == 'GB94':
        df["soil_respiration_gC_m2_yr"] = 0.75 * df["MODIS_NPP_gC_m2_yr"]

    co2_flux = df["soil_respiration_gC_m2_yr"].to_numpy() / MOLAR_MASS_C / SECONDS_PER_YEAR  # mol CO2 m-2 s-1

    BETA_J96 = 0.961
    PROFILE_DEPTH_M = 4.0
    ROOT_DEPTH_M = 1.35  # Rooting depth, from FAO-56 Table 22 for maize
    DZ_M = 0.01

    # Cell geometry
    z_edges_m = np.arange(0, ROOT_DEPTH_M + DZ_M, DZ_M)
    z_top_m, z_bottom_m = z_edges_m[:-1], z_edges_m[1:]
    z_mid_m = 0.5 * (z_top_m + z_bottom_m)
    dz_m = z_bottom_m - z_top_m

    # Jackson et al. (1996) root fraction in each layer; depth must be in cm
    root_fraction = (BETA_J96 ** (100 * z_top_m) - BETA_J96 ** (100 * z_bottom_m))
    root_fraction /= root_fraction.sum()

    # Intrinsic zero-order rate: mol L_bulk-1 s-1
    k_co2_root = co2_flux[:, None] * root_fraction[None, :] / (1000 * dz_m[None, :])

    # Pad k_co2 to the total profile_depth
    ns, root_nz = k_co2_root.shape
    nz = int(PROFILE_DEPTH_M / DZ_M) + 1
    k_co2 = np.zeros((ns, nz), dtype=np.float64)
    k_co2[:, :root_nz] = k_co2_root

    # Check that the depth-integrated rate recovers total respiration
    recovered = (k_co2 * 1000 * DZ_M).sum(axis=1) * SECONDS_PER_YEAR * MOLAR_MASS_C

    np.testing.assert_allclose(recovered, df["soil_respiration_gC_m2_yr"])

    # Save to S3
    OUTPUT_NPY = f"{s3_base_path}/input-data/processed-data/soilco2production_profiles_{soil_resp_calc}.npy"
    with fsspec.open(OUTPUT_NPY, "wb") as f:
        np.save(f, k_co2, allow_pickle=False)

### Plot the profiles across all sites

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 7), sharex=True, sharey=True, constrained_layout=True)

for i, ax in enumerate(axes.flat):
    ax.plot(k_co2[i], np.arange(0, PROFILE_DEPTH_M+DZ_M, DZ_M), color='g')
    ax.set(title=df.index[i], ylim=[4.00, 0.01], yticks=[0, 1, 2, 3, 4])

fig.supxlabel(r"CO$_2$ production rate (mol/L aquifer/s)")
fig.supylabel("Depth (m)")

## Plot: results — computed soil CO2 across sites

In [ ]:
pad = 5.0
extent = [
    df["lon"].min() - pad, df["lon"].max() + pad,
    df["lat"].min() - pad, df["lat"].max() + pad,
]

fig, ax = plt.subplots(figsize=(9, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent(extent, crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)

sc = ax.scatter(
    df["lon"], df["lat"], c=np.log10(df["soil_respiration_gC_m2_yr"]),
    cmap="viridis", s=40, transform=ccrs.PlateCarree(), zorder=3,
)
for i, site in enumerate(all_sites):
    ax.text(df.loc[site, 'lon']+0.5, df.loc[site, 'lat']+0.5, site, fontsize=10, transform=ccrs.PlateCarree())

plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.02, shrink=0.7, label="Soil CO₂ [log₁₀ gC/m2/yr]")
ax.set_title("Computed soil CO₂ by site")